### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [15]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Updated import
from pathlib import Path

In [16]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: A_Work_in_Progress_Context_based_Encryption_Scheme.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/BNJPCP+Times-Roman', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(43, 0, 2422949461744), '/LastChar': 169, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 333, 333, 0, 0, 0, 333, 250, 278, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 278, 0, 0, 0, 0, 0, 0, 0, 667, 667, 722, 611, 0, 0, 0, 0, 0, 0, 0, 0, 722, 0, 556, 0, 0, 556, 611, 0, 722, 0, 0, 722, 0, 0, 0, 0, 0, 0, 0, 444, 500, 444, 500, 444, 333, 500, 500, 278, 278, 0, 278, 778, 500, 500, 500, 0, 333, 389, 278, 500, 500, 722, 0, 500, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 500, 0, 0, 0, 0, 0, 0, 0, 0, 0, 250, 0, 0, 0, 0, 0, 0, 0, 0, 760]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary

  ✓ Loaded 5 pages

Total documents loaded: 5


In [17]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2015-07-31T01:12:00+05:30', 'author': 'Jungyub Lee', 'crossmarkdomains[1]': 'sciencedirect.com', 'crossmarkdomains[2]': 'elsevier.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'elsevierwebpdfspecifications': '6.4', 'moddate': '2015-07-31T02:34:53+05:30', 'subject': 'Procedia - Procedia Computer Science, 56 (2015) 271-275. doi:10.1016/j.procs.2015.07.208', 'title': 'A Work in Progress: Context based Encryption Scheme for Internet of Things', 'doi': '10.1016/j.procs.2015.07.208', 'robots': 'noindex', 'source': '..\\data\\pdf\\A_Work_in_Progress_Context_based_Encryption_Scheme.pdf', 'total_pages': 5, 'page': 0, 'page_label': '271', 'source_file': 'A_Work_in_Progress_Context_based_Encryption_Scheme.pdf', 'file_type': 'pdf'}, page_content='Procedia Computer Science   56  ( 2015 )  271 – 275 \nAvailable online at www.sciencedirect.com\n1877-0509

In [18]:
### Text splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [19]:
chunks = split_documents(all_pdf_documents)
chunks

Split 5 documents into 14 chunks

Example chunk:
Content: Procedia Computer Science   56  ( 2015 )  271 – 275 
Available online at www.sciencedirect.com
1877-0509 © 2015 Published by Elsevier B.V .This is an open access article under the CC BY-NC-ND license ...
Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2015-07-31T01:12:00+05:30', 'author': 'Jungyub Lee', 'crossmarkdomains[1]': 'sciencedirect.com', 'crossmarkdomains[2]': 'elsevier.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'elsevierwebpdfspecifications': '6.4', 'moddate': '2015-07-31T02:34:53+05:30', 'subject': 'Procedia - Procedia Computer Science, 56 (2015) 271-275. doi:10.1016/j.procs.2015.07.208', 'title': 'A Work in Progress: Context based Encryption Scheme for Internet of Things', 'doi': '10.1016/j.procs.2015.07.208', 'robots': 'noindex', 'source': '..\\data\\pdf\\A_Work_in_Progress_Context_based_Encryption_Scheme.pdf', 'total_page

[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2015-07-31T01:12:00+05:30', 'author': 'Jungyub Lee', 'crossmarkdomains[1]': 'sciencedirect.com', 'crossmarkdomains[2]': 'elsevier.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'elsevierwebpdfspecifications': '6.4', 'moddate': '2015-07-31T02:34:53+05:30', 'subject': 'Procedia - Procedia Computer Science, 56 (2015) 271-275. doi:10.1016/j.procs.2015.07.208', 'title': 'A Work in Progress: Context based Encryption Scheme for Internet of Things', 'doi': '10.1016/j.procs.2015.07.208', 'robots': 'noindex', 'source': '..\\data\\pdf\\A_Work_in_Progress_Context_based_Encryption_Scheme.pdf', 'total_pages': 5, 'page': 0, 'page_label': '271', 'source_file': 'A_Work_in_Progress_Context_based_Encryption_Scheme.pdf', 'file_type': 'pdf'}, page_content='Procedia Computer Science   56  ( 2015 )  271 – 275 \nAvailable online at www.sciencedirect.com\n1877-0509

### embedding and vectorStoreDB

In [20]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2305.37it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\DELL\AppData\Local\Temp\ipykernel_5640\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [22]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 42


In [23]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2015-07-31T01:12:00+05:30', 'author': 'Jungyub Lee', 'crossmarkdomains[1]': 'sciencedirect.com', 'crossmarkdomains[2]': 'elsevier.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'elsevierwebpdfspecifications': '6.4', 'moddate': '2015-07-31T02:34:53+05:30', 'subject': 'Procedia - Procedia Computer Science, 56 (2015) 271-275. doi:10.1016/j.procs.2015.07.208', 'title': 'A Work in Progress: Context based Encryption Scheme for Internet of Things', 'doi': '10.1016/j.procs.2015.07.208', 'robots': 'noindex', 'source': '..\\data\\pdf\\A_Work_in_Progress_Context_based_Encryption_Scheme.pdf', 'total_pages': 5, 'page': 0, 'page_label': '271', 'source_file': 'A_Work_in_Progress_Context_based_Encryption_Scheme.pdf', 'file_type': 'pdf'}, page_content='Procedia Computer Science   56  ( 2015 )  271 – 275 \nAvailable online at www.sciencedirect.com\n1877-0509

In [24]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 14 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.04s/it]


Generated embeddings with shape: (14, 384)
Adding 14 documents to vector store...
Successfully added 14 documents to vector store
Total documents in collection: 56


### Retriever Pipeline From VectorStore

In [25]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [26]:
rag_retriever

In [27]:
rag_retriever.retrieve("Context based Encryption Scheme")

Retrieving documents for query: 'Context based Encryption Scheme'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.97it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_e1761a78_10',
  'content': 'contexts are extracted by the detection method based on the user ’s situations. Therefore, context data can be used as \na suitable attribute value although user situation frequently changes. \nFig. 4. shows an example of context based encryption al gorithm. Encrypted data is generated by encrypting \ncontext and data. Data owner get a public key by a certificat ion agent and data is encrypted by the public key. When \ndata is decrypted, the certification agent creates a private key based on the user ’s context and provides the private \nkey to each user. A specific user can decrypt only the desired data by the key. \n4. Conclusion and Future Work \nTo enhance the security of Internet applications, some  researchers have proposed attribute based encryption \nmethod and access controls in cloud computing envir onments. However, a huge number of devices as a \ncharacteristic of IoT was not considered in the existing schemes. In th is paper, we hav

LLM

In [36]:
import os
from dotenv import find_dotenv, load_dotenv
from langchain_groq import ChatGroq

# 1. Automatically find and force-reload the .env file from the root directory
load_dotenv(find_dotenv(), override=True)

# 2. Extract API Key
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Please ensure it is added to your .env file in the root directory."
    )

# 3. Initialize Groq LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="gemma2-9b-it",
    temperature=0.1,
    max_tokens=1024
)

# 4. RAG Pipeline Function
def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve documents
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    
    if not context:
        return "No relevant context found to answer the question."
    
    # Prompt construct
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""
    
    # Pass prompt string directly into invoke
    response = llm.invoke(prompt)
    return response.content

In [37]:
answer = rag_simple("Context based Encryption Scheme",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Context based Encryption Scheme'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 11.46it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}